# Simple modelling notebook

This notebook performs a compact pipeline:

- Load processed day-averaged datasets for untreated monoculture, treated monoculture, and untreated coculture.
- Fit an ordinary logistic (no theta) to untreated monoculture and report `r`, `K`.
- For treated monoculture, fit two candidate models (multiplicative drug effect vs subtractive extra-death),
  using the untreated monoculture `r,K` as the baseline parameters (fixed for the treated fit).
  Select the best of those two by BIC and keep it for downstream steps.
- Fit a simple two-population coculture model (Option 1 style: S and R with logistic coupling and interaction term).

Notes:
- The notebook expects processed day-averages CSV files in the repository under `Processed_Datasets/*`.
- If the automatic file discovery doesn't find your CSVs, set `DATA_ROOT` and file paths manually below.

In [25]:
#= Package installer cell: will `using` each package and `Pkg.add` if missing =#
import Pkg
pkgs = ["CSV", "DataFrames", "DifferentialEquations", "Optim", "Distributions", "Plots", "StaticArrays", "ForwardDiff"]
for p in pkgs
    try
        @eval Main using $(Symbol(p))
    catch e
        println("Package " * p * " not available — installing...")
        Pkg.add(p)
        @eval Main using $(Symbol(p))
    end
end
println("Package installation step complete.")

Package installation step complete.


In [26]:
#= Cell 2: Imports and basic packages =#
using CSV, DataFrames, Statistics
using DifferentialEquations, Optim, Distributions
using Plots
gr(fmt=:png, dpi=120)

println("Loaded packages: CSV, DataFrames, DifferentialEquations, Optim, Plots")

Loaded packages: CSV, DataFrames, DifferentialEquations, Optim, Plots


In [27]:
#= Cell 3: Paths and helpers =#
# Robustly locate the repository `Processed_Datasets` directory by searching upward from the current working directory.
function locate_data_root()
    cur = pwd()
    while true
        candidate = joinpath(cur, "Processed_Datasets")
        if isdir(candidate)
            return candidate
        end
        parent = dirname(cur)
        if parent == cur
            break
        end
        cur = parent
    end
    # Fallback: assume Processed_Datasets sits one level above the current working directory's parent
    return joinpath(dirname(pwd()), "Processed_Datasets")
end
DATA_ROOT = locate_data_root()
println("Resolved DATA_ROOT = ", DATA_ROOT)

# --- Data loader helpers borrowed from the Untreated MonoCulture notebook ---
function _find_col(df::DataFrame; must_include::Vector{String}=String[])
    for c in names(df)
        s = lowercase(String(c))
        ok = all(substr -> occursin(substr, s), must_include)
        ok && return Symbol(c)
    end
    return nothing
end

function dedupe_xy(x::Vector{Float64}, y::Vector{Float64})
    df = DataFrame(day=x, value=y)
    g = combine(groupby(df, :day), :value => mean => :value)
    return Float64.(g.day), Float64.(g.value)
end

function load_monoculture_csv(path::AbstractString; scale_if_area::Float64=1.0)
    df = CSV.read(path, DataFrame)
    daycol = nothing
    if :day ∈ names(df)
        daycol = :day
    elseif :Day ∈ names(df)
        daycol = :Day
    else
        daycol = _find_col(df; must_include=["day"])
    end
    daycol === nothing && error("Could not find a 'day' column in $(basename(path))")
    preferred_exact = (
        :Cells, Symbol("Mean Cells"), Symbol("Average_Cells"), Symbol("Mean_Cells"),
    )
    valcol = nothing
    for cand in preferred_exact
        if cand ∈ names(df)
            valcol = cand
            break
        end
    end
    if valcol === nothing
        valcol = _find_col(df; must_include=["cells"])
    end
    if valcol === nothing
        valcol = _find_col(df; must_include=["mean","area"])
        if valcol === nothing && Symbol("Area µm^2") ∈ names(df)
            valcol = Symbol("Area µm^2")
        end
    end
    valcol === nothing && error("No suitable value column found in $(basename(path)). Looked for Cells/Mean Cells/Area.")
    x = Float64.(df[!, daycol])
    y = Float64.(df[!, valcol])
    valname = lowercase(String(valcol))
    if occursin("area", valname) && scale_if_area != 1.0
        y ./= scale_if_area
    end
    perm = sortperm(x)
    x = x[perm]; y = y[perm]
    return dedupe_xy(x, y)
end

function load_day_averages(path::AbstractString)
    df = CSV.read(path, DataFrame)
    daycol = if :Day ∈ names(df)
        :Day
    else
        _find_col(df; must_include=["day"]) |> x -> x === nothing ? error("No Day column in $(basename(path))") : x
    end
    valcol = if Symbol("Mean Cells") ∈ names(df)
        Symbol("Mean Cells")
    elseif Symbol("Mean_Cells") ∈ names(df)
        Symbol("Mean_Cells")
    else
        _find_col(df; must_include=["mean","cells"]) |> x -> x === nothing ? error("No mean-cells column in $(basename(path))") : x
    end
    yerr = nothing
    if Symbol("SEM Cells") ∈ names(df)
        yerr = Float64.(df[!, Symbol("SEM Cells")])
    elseif Symbol("SD Cells") ∈ names(df) && Symbol("N Samples") ∈ names(df)
        sd_col = Float64.(df[!, Symbol("SD Cells")])
        n_col  = Float64.(df[!, Symbol("N Samples")])
        yerr   = sd_col ./ sqrt.(n_col)
    elseif Symbol("Std_Error_Cells") ∈ names(df)
        yerr = Float64.(df[!, Symbol("Std_Error_Cells")])
    elseif Symbol("Std_Dev_Cells") ∈ names(df)
        yerr = Float64.(df[!, Symbol("Std_Dev_Cells")])
    elseif Symbol("SD Cells") ∈ names(df)
        yerr = Float64.(df[!, Symbol("SD Cells")])
    end
    x = Float64.(df[!, daycol])
    y = Float64.(df[!, valcol])
    perm = sortperm(x)
    x = x[perm]
    y = y[perm]
    yerr = yerr === nothing ? nothing : yerr[perm]
    return x, y, yerr
end

function infer_density(path::AbstractString)
    for part in reverse(splitpath(String(path)))
        lp = lowercase(part)
        if lp == "20k" || lp == "30k"
            return part
        end
    end
    return ""
end

function _infer_day0_cells(y::Vector{Float64}, density::String)
    if density == "30k"
        return 100.0
    elseif density == "20k"
        return 67.0
    else
        return max(y[1], eps())
    end
end

function load_day_averages_with_day0(path::AbstractString)
    x, y, yerr = load_day_averages(path)
    if yerr === nothing
        yerr = zeros(length(y))
    end
    if minimum(x) > 0.1
        density = infer_density(path)
        day0_cells = _infer_day0_cells(y, density)
        x = vcat([0.0], x); y = vcat([day0_cells], y); yerr = vcat([max(day0_cells*0.1,1.0)], yerr)
    end
    order = sortperm(x); return x[order], y[order], yerr[order]
end

# Hill effect helper
hill(dose, n, IC50) = dose^n / (IC50^n + dose^n)

Resolved DATA_ROOT = c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Processed_Datasets


hill (generic function with 1 method)

## 1) Fit untreated monoculture (classical logistic, v = 1)

We fit parameters r and K from an untreated monoculture CSV. The analytic solution for logistic (v=1) is used for speed:

In [28]:
#= Cell 5: analytic logistic solution and fitter =#
logistic_solution(t, r, K, N0) = K ./ (1 .+ ((K .- N0) ./ N0) .* exp.(-r .* t))

function fit_logistic(t, y, yerr; lower=[1e-4, 10.0], upper=[1.0, 1e6], nstarts=10)
    nobs = length(y)
    function loss_vec(p)
        r, K = p
        ypred = logistic_solution(t, r, K, y[1])
        w = 1.0 ./ (yerr.^2 .+ 1e-12)
        return sum(w .* (ypred .- y).^2)
    end

    best = (val=Inf, p=nothing, res=nothing)
    for s in 1:nstarts
        init = [rand(Uniform(l,u)) for (l,u) in zip(lower, upper)]
        res = Optim.optimize(loss_vec, lower, upper, init, Fminbox(NelderMead()); options=Optim.Options(iterations=400, show_trace=false))
        val = Optim.minimum(res)
        if val < best.val
            best = (val=val, p=Optim.minimizer(res), res=res)
        end
    end
    r̂, K̂ = best.p
    sse = best.val
    σ2 = sse / nobs
    bic = nobs * log(σ2) + 2 * log(nobs)  # k=2 params
    return (; r=r̂, K=K̂, sse=sse, bic=bic, nobs=nobs)
end


fit_logistic (generic function with 1 method)

In [30]:
#= Cell 6: Run untreated monoculture fit =#
# Try to auto-find an untreated monoculture example CSV path
# Optional: you can set `UNTREATED_MONO_OVERRIDE` (a full file path) before running this cell to bypass discovery.
if @isdefined(UNTREATED_MONO_OVERRIDE) && !isempty(UNTREATED_MONO_OVERRIDE)
    untreated_mono_path = UNTREATED_MONO_OVERRIDE
    println(\"Using manual override for untreated_mono_path: \", untreated_mono_path)
else
    untreated_mono_path = find_file_in_subdir(\"Untreated MonoCulture\", \"day_averages\")
end
if untreated_mono_path === nothing
    println(\"Could not automatically find an untreated monoculture CSV under Processed_Datasets/Untreated MonoCulture.\")
    println(\"Please set `UNTREATED_MONO_OVERRIDE` to your file path (for example: C:\\\\Users\\you\\...\\A2780Naive.csv) and re-run this cell.\")
    # Diagnostic info to help you locate the file
    println(\"Resolved DATA_ROOT = \", DATA_ROOT)
    udir = joinpath(DATA_ROOT, \"Untreated MonoCulture\")
    if isdir(udir)
        println(\"Files in Untreated MonoCulture:\")
        for f in readdir(udir)
            println(\"  - \", joinpath(udir, f))
        end
    else
        println(\"Processed_Datasets subdirectories:\")
        for d in readdir(DATA_ROOT)
            println(\"  - \", d)
        end
    end
else
    println(\"Using file: \", untreated_mono_path)
    t_u, y_u, yerr_u = load_day_averages_with_day0(untreated_mono_path)
    res_untreated = fit_logistic(t_u, y_u, yerr_u; nstarts=12)
    println(\"Untreated monoculture fit results:\")
    println(res_untreated)
    # Quick plot
    t_dense = collect(range(minimum(t_u), stop=maximum(t_u), length=200))
    ypred = logistic_solution(t_dense, res_untreated.r, res_untreated.K, y_u[1])
    scatter(t_u, y_u; yerror=yerr_u, label=\"data\")
    plot!(t_dense, ypred; label=\"logistic fit\", linewidth=2)
end

Base.Meta.ParseError: ParseError:
# Error @ c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\SimpleModels\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl:6:13
    untreated_mono_path = UNTREATED_MONO_OVERRIDE
    println(\"Using manual override for untreated_mono_path: \", untreated_mono_path)
#           ╙ ── not a unary operator

## 2) Treated monoculture: compare multiplicative vs subtractive drug models

We use the untreated `r,K` from the previous cell as baseline (fixed), and fit only the drug parameters for each candidate model. The two candidates implemented here are:

- Multiplicative: dN/dt = r * N * (1 - N/K) * (1 - E * hill(dose; n, IC50))
- Subtractive (extra-death per cell): dN/dt = r * N * (1 - N/K) - delta * hill(dose; n, IC50) * N

We compare by BIC (log-likelihood approximated from weighted SSE as in the untreated fit).

In [22]:
#= Cell 8: simulate and fit treated models (fix r,K from untreated) =#
function simulate_multiplicative(t, r, K, N0, E, n, IC50, dose)
    f!(du,u,p,t) = begin
        N = u[1]
        eff = 1.0 - E * hill(dose, n, IC50)
        du[1] = r * N * (1 - N / K) * eff
    end
    prob = ODEProblem(f!, [N0], (minimum(t), maximum(t)))
    sol = solve(prob; saveat=t, reltol=1e-6, abstol=1e-8)
    return Float64.(getindex.(sol.u,1))
end

function simulate_subtractive(t, r, K, N0, delta, n, IC50, dose)
    f!(du,u,p,t) = begin
        N = u[1]
        extra_death = delta * hill(dose, n, IC50)
        du[1] = r * N * (1 - N / K) - extra_death * N
    end
    prob = ODEProblem(f!, [N0], (minimum(t), maximum(t)))
    sol = solve(prob; saveat=t, reltol=1e-6, abstol=1e-8)
    return Float64.(getindex.(sol.u,1))
end

function fit_treated_fixed_rK(t, y, yerr, r_fixed, K_fixed, dose; nstarts=8, IC50=1.0)
    y0 = y[1]; nobs = length(y)

    # Multiplicative params: E, n
    function loss_mult(p)
        E, n = clamp.(p, [0.0, 0.1], [1.5, 6.0])
        ypred = try simulate_multiplicative(t, r_fixed, K_fixed, y0, E, n, IC50, dose) catch return 1e12 end
        w = 1.0 ./ (yerr.^2 .+ 1e-12)
        return sum(w .* (ypred .- y).^2)
    end

    best_mult = (val=Inf, p=nothing)
    for s in 1:nstarts
        init = [rand(Uniform(0.0,0.8)), rand(Uniform(0.5,3.0))]
        res = Optim.optimize(loss_mult, [0.0,0.1], [1.5,6.0], init, Fminbox(NelderMead()); options=Optim.Options(iterations=300))
        if Optim.minimum(res) < best_mult.val
            best_mult = (val=Optim.minimum(res), p=Optim.minimizer(res))
        end
    end
    k_mult = 2; σ2_mult = best_mult.val / nobs; bic_mult = nobs * log(σ2_mult) + k_mult * log(nobs)

    # Subtractive params: delta, n
    function loss_sub(p)
        delta, n = clamp.(p, [0.0, 0.1], [2.0, 6.0])
        ypred = try simulate_subtractive(t, r_fixed, K_fixed, y0, delta, n, IC50, dose) catch return 1e12 end
        w = 1.0 ./ (yerr.^2 .+ 1e-12)
        return sum(w .* (ypred .- y).^2)
    end
    best_sub = (val=Inf, p=nothing)
    for s in 1:nstarts
        init = [rand(Uniform(0.0,0.5)), rand(Uniform(0.5,3.0))]
        res = Optim.optimize(loss_sub, [0.0,0.1], [2.0,6.0], init, Fminbox(NelderMead()); options=Optim.Options(iterations=300))
        if Optim.minimum(res) < best_sub.val
            best_sub = (val=Optim.minimum(res), p=Optim.minimizer(res))
        end
    end
    k_sub = 2; σ2_sub = best_sub.val / nobs; bic_sub = nobs * log(σ2_sub) + k_sub * log(nobs)

    if bic_mult < bic_sub
        chosen = (:multiplicative, (E=best_mult.p[1], n=best_mult.p[2]), best_mult.val, bic_mult)
    else
        chosen = (:subtractive, (delta=best_sub.p[1], n=best_sub.p[2]), best_sub.val, bic_sub)
    end
    return (mult=(bic_mult, best_mult), sub=(bic_sub, best_sub), chosen=chosen)
end


Base.Meta.ParseError: ParseError:
# Error @ c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\SimpleModels\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:30:93
        E, n = clamp.(p, [0.0, 0.1], [1.5, 6.0])
        ypred = try simulate_multiplicative(t, r_fixed, K_fixed, y0, E, n, IC50, dose) catch return 1e12 end
#                                                                                           └──────────┘ ── a variable name is expected after `catch`

In [23]:
#= Cell 9: Run treated monoculture fits =#
treated_mono_path = find_file_in_subdir("Treated MonoCulture", "day_averages")
if treated_mono_path === nothing
    println("Could not automatically find a treated monoculture CSV under Processed_Datasets/Treated MonoCulture.")
    println("Please set `treated_mono_path` manually and re-run this cell.")
elseif !(@isdefined res_untreated)
    println("Please run the untreated monoculture cell first to obtain baseline r,K.")
else
    println("Using treated file: ", treated_mono_path)
    t_t, y_t, yerr_t = load_day_averages_with_day0(treated_mono_path)
    dose_val = 1.0  # replace with actual dose for that file if available (IC50 example). Adjust manually if different.
    res_treated = fit_treated_fixed_rK(t_t, y_t, yerr_t, res_untreated.r, res_untreated.K, dose_val; nstarts=12, IC50=1.0)
    println("Treated model comparison summary:")
    println("Multiplicative BIC: ", res_treated.mult[1])
    println("Subtractive BIC:   ", res_treated.sub[1])
    println("Chosen model (by BIC): ", res_treated.chosen)
end

Could not automatically find a treated monoculture CSV under Processed_Datasets/Treated MonoCulture.
Please set `treated_mono_path` manually and re-run this cell.


┌ Warning: Directory not found: c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\Processed_Datasets\Treated MonoCulture
└ @ Main c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\SimpleModels\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W2sZmlsZQ==.jl:11


## 3) Untreated coculture (Option 1 style)

We implement a simple two-population model where S and R have logistic growth with shared carrying capacity and interaction terms proportional to S*R (as in the image). We'll fix `K` to the untreated monoculture `K` and fit `r_s, r_r, alpha, beta`.

In [17]:
#= Cell 11: Coculture model and fitter =#
function simulate_coculture(t, r_s, r_r, K, alpha, beta, S0, R0)
    function f!(du,u,p,t)
        S = u[1]; R = u[2]; N = S + R
        du[1] = r_s * (1 - N / K) * S - alpha * S * R
        du[2] = r_r * (1 - N / K) * R - beta  * S * R
    end
    prob = ODEProblem(f!, [S0,R0], (minimum(t), maximum(t)))
    sol = solve(prob; saveat=t, reltol=1e-6, abstol=1e-8)
    return [Float64.(getindex.(sol.u,i)) for i in 1:2]  # (S_pred, R_pred) arrays
end

function fit_coculture(t, S_obs, R_obs, S_err, R_err; K_fixed, nstarts=10)
    nobs = length(t)
    function loss(p)
        r_s, r_r, alpha, beta, fracS0 = p
        # initial fraction to split starting total into S0,R0 (use observed at t=1 if available)
        S0 = S_obs[1]; R0 = R_obs[1]
        S_pred, R_pred = simulate_coculture(t, r_s, r_r, K_fixed, alpha, beta, S0, R0)
        wS = 1.0 ./ (S_err.^2 .+ 1e-12); wR = 1.0 ./ (R_err.^2 .+ 1e-12)
        return sum(wS .* (S_pred .- S_obs).^2) + sum(wR .* (R_pred .- R_obs).^2)
    end
    best = (val=Inf, p=nothing)
    lower = [1e-4, 1e-4, 0.0, 0.0, 0.0]; upper = [1.0, 1.0, 2.0, 2.0, 1.0]
    for s in 1:nstarts
        init = [rand(Uniform(l,u)) for (l,u) in zip(lower, upper)]
        res = Optim.optimize(loss, lower, upper, init, Fminbox(NelderMead()); options=Optim.Options(iterations=400))
        if Optim.minimum(res) < best.val
            best = (val=Optim.minimum(res), p=Optim.minimizer(res))
        end
    end
    return best
end


ErrorException: syntax: invalid keyword argument syntax "Optim.Options(iterations = 400)" around c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\SimpleModels\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:27

In [18]:
#= Cell 12: Run coculture fit =#
coculture_path = find_file_in_subdir("Untreated CoCulture", "day_averages")
if coculture_path === nothing
    println("Could not find an untreated coculture CSV automatically under Processed_Datasets/Untreated CoCulture. Set `coculture_path` manually and re-run.")
elseif !(@isdefined res_untreated)
    println("Please run the untreated monoculture cell first to obtain `K`.")
else
    println("Using coculture file: ", coculture_path)
    tc, yc_total, yc_err = load_day_averages_with_day0(coculture_path)
    # Attempt to recover S and R separate columns if the coculture CSV contains them (fall back to splitting total equally)
    dfc = CSV.read(coculture_path, DataFrame)
    names_l = lowercase.(String.(names(dfc)))
    s_idx = findfirst(x->occursin("s", x) && occursin("mean", x), names_l)
    r_idx = findfirst(x->occursin("r", x) && occursin("mean", x), names_l)
    if s_idx !== nothing && r_idx !== nothing
        S_obs = Float64.(dfc[!, s_idx]); R_obs = Float64.(dfc[!, r_idx])
        Serr = zeros(length(S_obs)); Rerr = zeros(length(R_obs))
    else
        # split total into two equal populations as a fallback (synthetic split) -- not ideal, user should provide separated S/R measurements
        S_obs = 0.5 .* yc_total; R_obs = 0.5 .* yc_total
        Serr = 0.5 .* yc_err; Rerr = 0.5 .* yc_err
        println("Warning: coculture file did not contain distinct S/R columns. Using a synthetic 50/50 split for fitting.")
    end
    best_coc = fit_coculture(tc, S_obs, R_obs, Serr, Rerr; K_fixed=res_untreated.K, nstarts=12)
    println("Best coculture fit (loss, params):")
    println(best_coc)
end

#= Cell X: Batch run: fit untreated (20k/30k), treated, and coculture =#
println("Starting batch fit for densities and cell lines...")
# Optional overrides: allow a single path string or a Dict mapping keys like "A2780Naive_30k" to file paths
unt_override = @isdefined(UNTREATED_MONO_OVERRIDE) ? UNTREATED_MONO_OVERRIDE : nothing
treated_override = @isdefined(TREATED_MONO_OVERRIDE) ? TREATED_MONO_OVERRIDE : nothing
coc_override = @isdefined(COCULTURE_OVERRIDE) ? COCULTURE_OVERRIDE : nothing
day_files = find_file_recursive("day_averages")
if day_files === nothing
    println("No day-averages CSVs found under DATA_ROOT=", DATA_ROOT)
else
    println("Found $(length(day_files)) day-averages files.")
end

# Helper: find best match from list by substrings
function choose_match(files::Vector{String}, parts::Vector{String})
    for f in files
        ok = all(p -> occursin(lowercase(p), lowercase(f)), parts)
        if ok return f end
    end
    return nothing
end

results = Dict()
densities = ["20k", "30k"]
cell_names = ["A2780Naive", "A2780cis"]

# Run untreated logistic fits for each density and cell line
if day_files !== nothing
    for dens in densities
        for cname in cell_names
            parts = [dens, cname, "day_averages"]
            f = choose_match(day_files, parts)
            if f === nothing
                println("No untreated day-averages found for ", cname, " (", dens, ").")
                continue
            end
            println("Fitting untreated: ", cname, " (", dens, "): ", f)
            x, y, yerr = load_day_averages_with_day0(f)
            rK = fit_logistic(x, y, yerr; nstarts=8)
            results[("untreated", dens, cname)] = (path=f, res=rK)
            println(" -> r=", rK.r, ", K=", rK.K)
        end
    end
end

# Run treated fits matching the untreated r,K when possible
treated_files = find_file_recursive("Treated MonoCulture")
if treated_files === nothing
    treated_files = find_file_recursive("day_averages")  # fallback to any day averages
end
if treated_files !== nothing
    for dens in densities
        for cname in cell_names
            # try to find a treated file matching density and cell name
            parts = ["Treated MonoCulture", dens, cname, "day_averages"]
            f = choose_match(treated_files, parts)
            if f === nothing
                # try looser match: any treated file with cell name
                f = choose_match(treated_files, [cname, "day_averages"])
            end
            if f === nothing
                println("No treated file found for ", cname, " (", dens, "). Skipping treated fit.")
                continue
            end
            key = ("untreated", dens, cname)
            if !haskey(results, key)
                println("No untreated fit available for ", cname, " (", dens, "). Cannot fix r,K. Skipping treated fit.")
                continue
            end
            rK = results[key].res
            println("Fitting treated (fixed r,K) for ", cname, " (", dens, "): ", f)
            tt, yy, ye = load_day_averages_with_day0(f)
            res_t = fit_treated_fixed_rK(tt, yy, ye, rK.r, rK.K, 1.0; nstarts=8, IC50=1.0)
            results[("treated", dens, cname)] = (path=f, res=res_t)
            println(" -> treated chosen model: ", res_t.chosen[1])
        end
    end
end

# Run coculture fits (use matching density K if available)
coc_files = find_file_recursive("Untreated CoCulture")
if coc_files === nothing
    coc_files = find_file_recursive("co")  # fallback
end
if coc_files !== nothing
    for f in coc_files
        println("Fitting coculture file: ", f)
        tc, yc_total, yc_err = load_day_averages_with_day0(f)
        # try to pick K from untreated with same density
        matchedK = nothing
        for dens in densities
            for cname in cell_names
                key = ("untreated", dens, cname)
                if haskey(results, key) && occursin(lowercase(dens), lowercase(f))
                    matchedK = results[key].res.K; break
                end
            end
            if matchedK !== nothing break end
        end
        if matchedK === nothing && !isempty(values(results))
            # fallback to first untreated K
            for v in values(results)
                if haskey(v, :res) && typeof(v.res) <: NamedTuple
                    matchedK = v.res.K; break
                elseif haskey(v, :res) && hasproperty(v.res, :K)
                    matchedK = v.res.K; break
                end
            end
        end
        if matchedK === nothing
            println("No K available for coculture fit; skipping file: ", f); continue
        end
        dfc = CSV.read(f, DataFrame)
        names_l = lowercase.(String.(names(dfc)))
        s_idx = findfirst(x->occursin("s", x) && occursin("mean", x), names_l)
        r_idx = findfirst(x->occursin("r", x) && occursin("mean", x), names_l)
        if s_idx !== nothing && r_idx !== nothing
            S_obs = Float64.(dfc[!, s_idx]); R_obs = Float64.(dfc[!, r_idx])
            Serr = zeros(length(S_obs)); Rerr = zeros(length(R_obs))
        else
            S_obs = 0.5 .* yc_total; R_obs = 0.5 .* yc_total
            Serr = 0.5 .* yc_err; Rerr = 0.5 .* yc_err
            println("Warning: using synthetic 50/50 split for coculture file: ", f)
        end
        bestc = fit_coculture(tc, S_obs, R_obs, Serr, Rerr; K_fixed=matchedK, nstarts=8)
        results[("coculture", f)] = (path=f, res=bestc)
        println(" -> coculture loss=", bestc.val)
    end
end

println("Batch fitting complete. Summary of results keys:")
for k in keys(results) println("  - ", k) end

Could not find an untreated coculture CSV automatically under Processed_Datasets/Untreated CoCulture. Set `coculture_path` manually and re-run.


┌ Warning: Directory not found: c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\Processed_Datasets\Untreated CoCulture
└ @ Main c:\Users\elbak\Desktop\ForgeLabs\CancerGrowthDynamics\Modelling Data Notebooks\SimpleModels\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W2sZmlsZQ==.jl:11


## Save / report results

You can extract the parameter estimates from `res_untreated`, `res_treated`, and `best_coc` and save them to CSV or return them for downstream analysis.

In [19]:
#= Cell 14: Example export (if results exist) =#
results = Dict()
if @isdefined(res_untreated)
    results["untreated_monoculture"] = (r=res_untreated.r, K=res_untreated.K, sse=res_untreated.sse, bic=res_untreated.bic)
end
if @isdefined(res_treated)
    results["treated_comparison"] = res_treated
end
if @isdefined(best_coc)
    results["coculture"] = best_coc
end

if !isempty(results)
    df = DataFrame(Key=String[], Value=String[])
    for (k,v) in results
        push!(df, (String(k), string(v)))
    end
    CSV.write("simple_model_results_summary.csv", df)
    println("Wrote simple_model_results_summary.csv")
else
    println("No results to save yet. Run the fitting cells first.")
end

No results to save yet. Run the fitting cells first.
